# Notebook 11 - Grafana Monitoring Dashboard
**Project**: Real-Time Retail Analytics and Demand Prediction Platform

**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar

Sets up Prometheus metrics endpoint and configures Grafana dashboard with system metrics.

Grafana URL: **http://localhost:3000** (admin/admin)

## 11.1 Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install prometheus_client flask requests psutil -q
print('Done.')

Done.


## 11.2 Create Prometheus Metrics Endpoint
This exposes retail pipeline metrics that Prometheus scrapes and Grafana visualizes.

In [2]:
from prometheus_client import Gauge, Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST
from flask import Flask, Response
import threading
import time
import numpy as np
import psutil

# Define metrics
kafka_messages_total    = Counter('kafka_messages_total', 'Total messages streamed via Kafka')
kafka_throughput        = Gauge('kafka_throughput_per_sec', 'Kafka messages per second')
delta_lake_records      = Gauge('delta_lake_records_total', 'Total records in Delta Lake')
api_requests_total      = Counter('api_requests_total', 'Total API prediction requests')
api_latency             = Histogram('api_latency_seconds', 'API response latency', buckets=[0.01, 0.05, 0.1, 0.25, 0.5, 1.0])
model_prediction_value  = Gauge('model_prediction_value', 'Latest model prediction value')
pipeline_revenue_total  = Gauge('pipeline_revenue_total', 'Total revenue processed')
pipeline_customers      = Gauge('pipeline_customers_total', 'Total unique customers')
pipeline_products       = Gauge('pipeline_products_total', 'Total unique products')
pipeline_countries      = Gauge('pipeline_countries_total', 'Total countries served')
system_cpu_percent      = Gauge('system_cpu_percent', 'System CPU usage percent')
system_memory_percent   = Gauge('system_memory_percent', 'System memory usage percent')
system_disk_percent     = Gauge('system_disk_percent', 'System disk usage percent')
spark_jobs_completed    = Gauge('spark_jobs_completed', 'Number of Spark jobs completed')
ml_model_r2_score       = Gauge('ml_model_r2_score', 'ML model R2 score')
ml_model_rmse           = Gauge('ml_model_rmse', 'ML model RMSE')

# Set initial static values from pipeline
delta_lake_records.set(500000)
pipeline_revenue_total.set(10881574.77)
pipeline_customers.set(4649)
pipeline_products.set(4045)
pipeline_countries.set(40)
spark_jobs_completed.set(4)
ml_model_r2_score.set(0.9289)
ml_model_rmse.set(87.97)

# Background thread to update dynamic metrics
def update_metrics():
    msg_count = 0
    while True:
        # Simulate Kafka streaming
        batch = np.random.randint(200, 500)
        msg_count += batch
        kafka_messages_total.inc(batch)
        kafka_throughput.set(np.random.randint(250, 450))
        
        # Simulate API requests
        api_requests_total.inc(np.random.randint(1, 5))
        api_latency.observe(np.random.uniform(0.005, 0.05))
        model_prediction_value.set(round(np.random.uniform(50, 300), 2))
        
        # System metrics
        system_cpu_percent.set(psutil.cpu_percent())
        system_memory_percent.set(psutil.virtual_memory().percent)
        system_disk_percent.set(psutil.disk_usage('/').percent)
        
        time.sleep(5)

threading.Thread(target=update_metrics, daemon=True).start()

# Flask app to serve metrics
metrics_app = Flask('metrics')

@metrics_app.route('/metrics')
def metrics():
    return Response(generate_latest(), mimetype=CONTENT_TYPE_LATEST)

@metrics_app.route('/')
def home():
    return '<h3>Retail Pipeline Metrics</h3><p><a href="/metrics">View Prometheus Metrics</a></p>'

def run_metrics():
    metrics_app.run(host='0.0.0.0', port=8099, debug=False, use_reloader=False)

threading.Thread(target=run_metrics, daemon=True).start()
time.sleep(2)

print('Metrics endpoint running at http://localhost:8099/metrics')
print('Prometheus can now scrape this endpoint.')

 * Serving Flask app 'metrics'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8099
 * Running on http://172.18.0.10:8099
Press CTRL+C to quit
172.18.0.2 - - [28/Apr/2026 21:06:47] "GET /metrics HTTP/1.1" 200 -


Metrics endpoint running at http://localhost:8099/metrics
Prometheus can now scrape this endpoint.


172.18.0.2 - - [28/Apr/2026 21:06:52] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:06:57] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:02] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:07] "GET /metrics HTTP/1.1" 200 -
172.18.0.1 - - [28/Apr/2026 21:07:08] "GET /metrics HTTP/1.1" 200 -
172.18.0.1 - - [28/Apr/2026 21:07:08] "GET /favicon.ico HTTP/1.1" 404 -
172.18.0.2 - - [28/Apr/2026 21:07:12] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:17] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:07:19] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:22] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:27] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:32] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:37] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:42] "GET /metrics HTTP/1.1" 200 -
172.18.0.2 - - [28/Apr/2026 21:07:47] "GET /m

## 11.3 Verify Metrics Endpoint

In [3]:
import requests

resp = requests.get('http://localhost:8099/metrics', timeout=5)
print(f'Status: {resp.status_code}')
print()
# Show key metrics
for line in resp.text.split('\n'):
    if line and not line.startswith('#') and any(k in line for k in ['kafka_', 'delta_', 'api_', 'pipeline_', 'system_', 'spark_', 'ml_model_']):
        print(f'  {line}')

Status: 200

  kafka_messages_total 2444.0
  kafka_messages_created 1.7774104067276745e+09
  kafka_throughput_per_sec 423.0
  delta_lake_records_total 500000.0
  api_requests_total 19.0
  api_requests_created 1.7774104067278368e+09
  api_latency_seconds_bucket{le="0.01"} 0.0
  api_latency_seconds_bucket{le="0.05"} 7.0
  api_latency_seconds_bucket{le="0.1"} 7.0
  api_latency_seconds_bucket{le="0.25"} 7.0
  api_latency_seconds_bucket{le="0.5"} 7.0
  api_latency_seconds_bucket{le="1.0"} 7.0
  api_latency_seconds_bucket{le="+Inf"} 7.0
  api_latency_seconds_count 7.0
  api_latency_seconds_sum 0.22000408450681616
  api_latency_seconds_created 1.777410406727904e+09
  pipeline_revenue_total 1.088157477e+07
  pipeline_customers_total 4649.0
  pipeline_products_total 4045.0
  pipeline_countries_total 40.0
  system_cpu_percent 5.1
  system_memory_percent 52.4
  system_disk_percent 2.7
  spark_jobs_completed 4.0
  ml_model_r2_score 0.9289
  ml_model_rmse 87.97


## 11.4 Update Prometheus Config to Scrape Our Metrics
We need to tell Prometheus (running in Docker) to scrape our metrics endpoint.

In [4]:
prometheus_config = '''global:
  scrape_interval: 10s
  evaluation_interval: 10s

scrape_configs:
  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']

  - job_name: 'retail-pipeline'
    scrape_interval: 5s
    static_configs:
      - targets: ['jupyter:8099']
        labels:
          project: 'z5008-retail'
          team: 'vineet-joshi'
'''

with open('/home/jovyan/work/prometheus.yml', 'w') as f:
    f.write(prometheus_config)

print('prometheus.yml created at /home/jovyan/work/prometheus.yml')
print()
print('IMPORTANT: Copy this file to your config/ folder and restart Prometheus:')
print('  1. Copy prometheus.yml to ./config/prometheus.yml on your host')
print('  2. Run: docker-compose restart prometheus')
print('  3. Verify at http://localhost:9090/targets')

prometheus.yml created at /home/jovyan/work/prometheus.yml

IMPORTANT: Copy this file to your config/ folder and restart Prometheus:
  1. Copy prometheus.yml to ./config/prometheus.yml on your host
  2. Run: docker-compose restart prometheus
  3. Verify at http://localhost:9090/targets


## 11.5 Create Grafana Dashboard JSON

In [5]:
import json

dashboard = {
    "dashboard": {
        "id": None,
        "uid": "retail-pipeline-v2",
        "title": "Retail Analytics Pipeline - Z5008",
        "tags": ["retail", "z5008", "bigdata"],
        "timezone": "browser",
        "refresh": "5s",
        "time": {"from": "now-30m", "to": "now"},
        "panels": [
            {
                "id": 1,
                "title": "Kafka Messages Total",
                "type": "stat",
                "gridPos": {"h": 4, "w": 4, "x": 0, "y": 0},
                "targets": [{"expr": "kafka_messages_total", "legendFormat": "Messages"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "green"}, "thresholds": {"mode": "absolute", "steps": [{"value": None, "color": "green"}]}}}
            },
            {
                "id": 2,
                "title": "Kafka Throughput (msg/sec)",
                "type": "gauge",
                "gridPos": {"h": 4, "w": 4, "x": 4, "y": 0},
                "targets": [{"expr": "kafka_throughput_per_sec", "legendFormat": "Throughput"}],
                "fieldConfig": {"defaults": {"min": 0, "max": 500, "thresholds": {"mode": "absolute", "steps": [{"value": None, "color": "green"}, {"value": 300, "color": "yellow"}, {"value": 400, "color": "red"}]}}}
            },
            {
                "id": 3,
                "title": "Delta Lake Records",
                "type": "stat",
                "gridPos": {"h": 4, "w": 4, "x": 8, "y": 0},
                "targets": [{"expr": "delta_lake_records_total", "legendFormat": "Records"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "blue"}}}
            },
            {
                "id": 4,
                "title": "API Requests Total",
                "type": "stat",
                "gridPos": {"h": 4, "w": 4, "x": 12, "y": 0},
                "targets": [{"expr": "api_requests_total", "legendFormat": "Requests"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "purple"}}}
            },
            {
                "id": 5,
                "title": "ML Model R2 Score",
                "type": "stat",
                "gridPos": {"h": 4, "w": 4, "x": 16, "y": 0},
                "targets": [{"expr": "ml_model_r2_score", "legendFormat": "R2"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "orange"}, "decimals": 4}}
            },
            {
                "id": 6,
                "title": "Revenue Processed",
                "type": "stat",
                "gridPos": {"h": 4, "w": 4, "x": 20, "y": 0},
                "targets": [{"expr": "pipeline_revenue_total", "legendFormat": "Revenue"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "green"}, "unit": "currencyGBP"}}
            },
            {
                "id": 7,
                "title": "Kafka Throughput Over Time",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 12, "x": 0, "y": 4},
                "targets": [{"expr": "kafka_throughput_per_sec", "legendFormat": "Messages/sec"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "green"}, "custom": {"fillOpacity": 20, "lineWidth": 2}}}
            },
            {
                "id": 8,
                "title": "System CPU & Memory",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 12, "x": 12, "y": 4},
                "targets": [
                    {"expr": "system_cpu_percent", "legendFormat": "CPU %"},
                    {"expr": "system_memory_percent", "legendFormat": "Memory %"}
                ],
                "fieldConfig": {"defaults": {"unit": "percent", "min": 0, "max": 100, "custom": {"fillOpacity": 10, "lineWidth": 2}}}
            },
            {
                "id": 9,
                "title": "API Latency Distribution",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 8, "x": 0, "y": 12},
                "targets": [{"expr": "rate(api_latency_seconds_sum[1m]) / rate(api_latency_seconds_count[1m])", "legendFormat": "Avg Latency"}],
                "fieldConfig": {"defaults": {"unit": "s", "color": {"mode": "fixed", "fixedColor": "yellow"}, "custom": {"fillOpacity": 15}}}
            },
            {
                "id": 10,
                "title": "Kafka Messages Rate",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 8, "x": 8, "y": 12},
                "targets": [{"expr": "rate(kafka_messages_total[1m])", "legendFormat": "Rate"}],
                "fieldConfig": {"defaults": {"color": {"mode": "fixed", "fixedColor": "blue"}, "custom": {"fillOpacity": 20}}}
            },
            {
                "id": 11,
                "title": "Pipeline Overview",
                "type": "stat",
                "gridPos": {"h": 8, "w": 8, "x": 16, "y": 12},
                "targets": [
                    {"expr": "pipeline_customers_total", "legendFormat": "Customers"},
                    {"expr": "pipeline_products_total", "legendFormat": "Products"},
                    {"expr": "pipeline_countries_total", "legendFormat": "Countries"},
                    {"expr": "spark_jobs_completed", "legendFormat": "Spark Jobs"}
                ],
                "fieldConfig": {"defaults": {"color": {"mode": "palette-classic"}}}
            }
        ],
        "schemaVersion": 38
    },
    "overwrite": True
}

# Save JSON
with open('/home/jovyan/work/grafana_dashboard.json', 'w') as f:
    json.dump(dashboard, f, indent=2)

print('Grafana dashboard JSON saved to /home/jovyan/work/grafana_dashboard.json')
print(f'Panels: {len(dashboard["dashboard"]["panels"])}')

Grafana dashboard JSON saved to /home/jovyan/work/grafana_dashboard.json
Panels: 11


## 11.6 Import Dashboard to Grafana via API

In [6]:
import requests
import json

GRAFANA_URL =  'http://172.18.0.1:3002'
GRAFANA_AUTH = ('admin', 'admin')

# Step 1: Add Prometheus as data source
datasource = {
    'name': 'Prometheus',
    'type': 'prometheus',
    'url': 'http://prometheus:9090',
    'access': 'proxy',
    'isDefault': True
}

try:
    resp = requests.post(
        f'{GRAFANA_URL}/api/datasources',
        json=datasource,
        auth=GRAFANA_AUTH,
        timeout=10
    )
    if resp.status_code == 200:
        print('Prometheus datasource added to Grafana.')
    elif resp.status_code == 409:
        print('Prometheus datasource already exists.')
    else:
        print(f'Datasource response: {resp.status_code} {resp.text[:200]}')
except Exception as e:
    print(f'Could not reach Grafana: {e}')
    print('Make sure Grafana is running: docker-compose up -d grafana')

Prometheus datasource already exists.


In [7]:
# Step 2: Import dashboard
try:
    with open('/home/jovyan/work/grafana_dashboard.json', 'r') as f:
        dash_json = json.load(f)

    resp = requests.post(
        f'{GRAFANA_URL}/api/dashboards/db',
        json=dash_json,
        auth=GRAFANA_AUTH,
        headers={'Content-Type': 'application/json'},
        timeout=10
    )

    if resp.status_code == 200:
        result = resp.json()
        print('Dashboard imported to Grafana!')
        print(f'  URL: http://localhost:3002{result.get("url", "/d/retail-pipeline-v2")}')
    else:
        print(f'Import response: {resp.status_code}')
        print(resp.text[:300])
except Exception as e:
    print(f'Error importing dashboard: {e}')
    print('\nManual import instructions:')
    print('  1. Open http://localhost:3000')
    print('  2. Login: admin / admin')
    print('  3. Go to Dashboards > Import')
    print('  4. Upload grafana_dashboard.json')

Dashboard imported to Grafana!
  URL: http://localhost:3002/d/retail-pipeline-v2/retail-analytics-pipeline-z5008


## 11.7 Verify Grafana Setup

In [8]:
# Check Grafana health
try:
    resp = requests.get(f'{GRAFANA_URL}/api/health', timeout=5)
    print(f'Grafana Health: {resp.json()}')
except:
    print('Grafana not reachable. Run: docker-compose up -d grafana')

# Check datasources
try:
    resp = requests.get(f'{GRAFANA_URL}/api/datasources', auth=GRAFANA_AUTH, timeout=5)
    print(f'\nDatasources:')
    for ds in resp.json():
        print(f"  {ds['name']} ({ds['type']}) -> {ds['url']}")
except:
    pass

# Check dashboards
try:
    resp = requests.get(f'{GRAFANA_URL}/api/search', auth=GRAFANA_AUTH, timeout=5)
    print(f'\nDashboards:')
    for d in resp.json():
        print(f"  {d['title']} -> http://localhost:3002{d['url']}")
except:
    pass

Grafana Health: {'database': 'ok', 'version': '13.0.1', 'commit': 'a100054f'}

Datasources:
  Prometheus (prometheus) -> http://prometheus:9090
  prometheus-1 (prometheus) -> http://prometheus:9090

Dashboards:
  Retail Analytics Pipeline - Z5008 -> http://localhost:3002/d/retail-pipeline-v2/retail-analytics-pipeline-z5008


## 11.8 Manual Setup Instructions (if API import fails)

In [9]:
print('='*60)
print('GRAFANA MANUAL SETUP (if API import did not work)')
print('='*60)
print()
print('Step 1: Open http://localhost:3000')
print('        Login: admin / admin')
print()
print('Step 2: Add Prometheus Data Source')
print('        Go to: Connections > Data Sources > Add')
print('        Select: Prometheus')
print('        URL: http://prometheus:9090')
print('        Click: Save & Test')
print()
print('Step 3: Import Dashboard')
print('        Go to: Dashboards > New > Import')
print('        Upload: grafana_dashboard.json')
print('        (file is at /home/jovyan/work/grafana_dashboard.json)')
print()
print('Step 4: Update Prometheus Config')
print('        Copy prometheus.yml to ./config/prometheus.yml on host')
print('        Run: docker-compose restart prometheus')
print('        Verify: http://localhost:9090/targets')
print()
print('Step 5: Create Panels Manually (if import fails)')
print('        Dashboard > Add Panel > Add Query')
print('        Use these PromQL queries:')
print()
print('        Panel 1 - Kafka Messages:    kafka_messages_total')
print('        Panel 2 - Kafka Throughput:   kafka_throughput_per_sec')
print('        Panel 3 - Delta Lake Records: delta_lake_records_total')
print('        Panel 4 - API Requests:       api_requests_total')
print('        Panel 5 - CPU Usage:          system_cpu_percent')
print('        Panel 6 - Memory Usage:       system_memory_percent')
print('        Panel 7 - API Latency:        rate(api_latency_seconds_sum[1m])')
print('        Panel 8 - ML Model R2:        ml_model_r2_score')
print('        Panel 9 - Revenue Total:      pipeline_revenue_total')
print('        Panel 10- Kafka Rate:          rate(kafka_messages_total[1m])')
print('        Panel 11- Customers:           pipeline_customers_total')

GRAFANA MANUAL SETUP (if API import did not work)

Step 1: Open http://localhost:3000
        Login: admin / admin

Step 2: Add Prometheus Data Source
        Go to: Connections > Data Sources > Add
        Select: Prometheus
        URL: http://prometheus:9090
        Click: Save & Test

Step 3: Import Dashboard
        Go to: Dashboards > New > Import
        Upload: grafana_dashboard.json
        (file is at /home/jovyan/work/grafana_dashboard.json)

Step 4: Update Prometheus Config
        Copy prometheus.yml to ./config/prometheus.yml on host
        Run: docker-compose restart prometheus
        Verify: http://localhost:9090/targets

Step 5: Create Panels Manually (if import fails)
        Dashboard > Add Panel > Add Query
        Use these PromQL queries:

        Panel 1 - Kafka Messages:    kafka_messages_total
        Panel 2 - Kafka Throughput:   kafka_throughput_per_sec
        Panel 3 - Delta Lake Records: delta_lake_records_total
        Panel 4 - API Requests:       api_

## 11.9 Summary

In [10]:
print('='*55)
print('NOTEBOOK 11 COMPLETE')
print('='*55)
print(f'  Metrics endpoint  : http://localhost:8099/metrics')
print(f'  Prometheus        : http://localhost:9090')
print(f'  Grafana           : http://localhost:3002 (admin/admin)')
print(f'  Dashboard         : Retail Analytics Pipeline - Z5008')
print(f'  Panels            : 11 (stats + gauges + timeseries)')
print(f'  Auto-refresh      : Every 5 seconds')
print()
print('Metrics tracked:')
print('  - Kafka messages total + throughput')
print('  - Delta Lake record count')
print('  - API request count + latency')
print('  - ML model R2 score + RMSE')
print('  - Pipeline revenue, customers, products, countries')
print('  - System CPU, memory, disk usage')
print('  - Spark jobs completed')
print()
print('Files created:')
print('  prometheus.yml')
print('  grafana_dashboard.json')

NOTEBOOK 11 COMPLETE
  Metrics endpoint  : http://localhost:8099/metrics
  Prometheus        : http://localhost:9090
  Grafana           : http://localhost:3002 (admin/admin)
  Dashboard         : Retail Analytics Pipeline - Z5008
  Panels            : 11 (stats + gauges + timeseries)
  Auto-refresh      : Every 5 seconds

Metrics tracked:
  - Kafka messages total + throughput
  - Delta Lake record count
  - API request count + latency
  - ML model R2 score + RMSE
  - Pipeline revenue, customers, products, countries
  - System CPU, memory, disk usage
  - Spark jobs completed

Files created:
  prometheus.yml
  grafana_dashboard.json


In [11]:
import json

with open('/home/jovyan/work/grafana_dashboard.json', 'r') as f:
    data = json.load(f)

# Extract just the dashboard part (remove the wrapper)
if 'dashboard' in data:
    dash_only = data['dashboard']
else:
    dash_only = data

# Save fixed version
with open('/home/jovyan/work/grafana_dashboard_import.json', 'w') as f:
    json.dump(dash_only, f, indent=2)

print('Fixed file saved: grafana_dashboard_import.json')
print('Upload this file to Grafana instead.')

Fixed file saved: grafana_dashboard_import.json
Upload this file to Grafana instead.


In [12]:
# Stop all background servers
import os, signal

# List running threads
import threading
print(f'Active threads: {threading.active_count()}')
for t in threading.enumerate():
    print(f'  {t.name}')

Active threads: 8
  MainThread
  IOPub
  Heartbeat
  Control
  IPythonHistorySavingThread
  Thread-2
  Thread-3 (update_metrics)
  Thread-4 (run_metrics)
